In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydhn
from pydhn.fluids import Fluid
from pydhn.soils import Soil
from pydhn.solving import SimpleStep, Scheduler

In [ ]:
# ---------------------------------------------------------------------------
# Simulation parameters
# ---------------------------------------------------------------------------
STEPSIZE       = 600          # time step [s]  (10 min)
NUM_SIM_STEPS  = 1008         # 7 days × 144 steps/day
T_AMBIENT_C    = 10.0         # ambient / ground temperature [°C]

In [ ]:
# ---------------------------------------------------------------------------
# File paths
# ---------------------------------------------------------------------------
PIPES_FILE = "pipes.csv"
NODES_FILE = "nodes.csv"
DEMAND_PROFILE_FILE = "../../data/demand_profile.csv"
MASSFLOW_SRC1_PROFILE = "src1_mdot_profile.csv"

# ---------------------------------------------------------------------------
# Load input data
# ---------------------------------------------------------------------------
pipe_data = pd.read_csv(PIPES_FILE)
node_data = pd.read_csv(NODES_FILE)
demand_profile_data = pd.read_csv(DEMAND_PROFILE_FILE)
demand_profile_data = demand_profile_data[:NUM_SIM_STEPS]


time_sim   = demand_profile_data["timestep"].values * STEPSIZE  # [s]
heat_demand = demand_profile_data["Q_W"] * (STEPSIZE / 3600)    # [Wh]
MDOT_BYPASS_KGS = 1.77 / 3600                                   # [kg/s], bypass mass flow

source_rows        = node_data[node_data["node_type"] == "source"].reset_index(drop=True)
src1_name          = source_rows.loc[1, "node_id"]
src1_profile_file  = source_rows.loc[1, "m_flow_set_kg_s"]

src1_profile_data = pd.read_csv(src1_profile_file)
src1_profile_data = src1_profile_data[:NUM_SIM_STEPS]
mdot_src1_ts      = src1_profile_data["mdot_kg_s"].values

print(f"Loaded topological data from {PIPES_FILE} "
      f"--- dynamic simulation of {len(demand_profile_data)} time steps.")

In [ ]:
# ---------------------------------------------------------------------------
# Fluid and soil
# ---------------------------------------------------------------------------
# Properties taken from first pipe row (homogeneous network assumption)
p0 = pipe_data.iloc[0]

fluid = Fluid(
    name       = "Water",
    isconstant = True,
    cp         = p0["fluid_cp_JkgK"],
    mu         = p0["fluid_mu_Pas"],
    rho        = p0["fluid_rho_kgm3"],
    k          = p0["fluid_lambda_WmK"],
)

soil = Soil(
    k    = p0["soil_lambda_WmK"],
    temp = p0["T_ground_degC"],
)

# ---------------------------------------------------------------------------
# Built network
# ---------------------------------------------------------------------------
net = pydhn.Network()

# Add nodes
for idx, row in node_data.iterrows():
    name      = row["node_id"]
    node_type = row["node_type"]
    X, Y      = row["x_m"], row["y_m"]

    name_s = f"{name}_s"
    name_r = f"{name}_r"

    net.add_node(name=name_s, x=X, y=-Y)
    net.add_node(name=name_r, x=X, y=-Y)

    # Add substations
    if node_type == "substation":
        delta_t = float(row["delta_T_nom_K"])
        net.add_consumer(
            name              = name,
            start_node        = name_s,
            end_node          = name_r,
            control_type      = "energy",
            setpoint_type_hx  = "delta_t",
            setpoint_value_hx = -delta_t,
            design_delta_t    = abs(delta_t),
            t_out_min_hx      = 20,
            mass_flow_min     = MDOT_BYPASS_KGS,
            stepsize          = STEPSIZE,
        )

    # Add central generators
    elif node_type == "source":
        t_supply    = float(row["T_supply_set_degC"])
        mdot_supply = mdot_src1_ts[0]
        p_ref       = float(row["p_pump_set_bar"])

        if idx == 0:
            setpoint_type_hyd  = "pressure"
            setpoint_value_hyd = - p_ref * 1e5
        else:
            setpoint_type_hyd  = "mass_flow"
            setpoint_value_hyd = mdot_supply

        net.add_producer(
            name                = name,
            start_node          = name_r,
            end_node            = name_s,
            setpoint_type_hyd   = setpoint_type_hyd,
            setpoint_value_hyd  = setpoint_value_hyd,
            setpoint_type_hx    = "t_out",
            setpoint_value_hx   = t_supply,
            stepsize            = STEPSIZE,
        )

# Add pipes
for _, row in pipe_data.iterrows():
    name = row["pipe_id"]
    frm  = row["from_node"]
    to   = row["to_node"]

    common = dict(
        # --- geometry ---
        diameter                = row["d_inner_m"],                 # carrier pipe inner diameter [m]
        length                  = row["length_m"],                  # pipe length [m]
        internal_pipe_thickness = row["wall_m"],                    # steel wall thickness [m]
        insulation_thickness    = row["insulation_thickness_m"],    # PUR foam thickness [m]
        casing_thickness        = row["casing_thickness_m"],        # HDPE casing wall thickness [m]
        # --- hydraulic ---
        roughness               = row["roughness_mm"],              # absolute roughness [mm]
        # --- thermal: insulation ---
        k_insulation            = row["PUR_lambda_WmK"],            # PUR thermal conductivity [W/(m·K)]
        # --- thermal: carrier pipe ---
        k_internal_pipe         = row["steel_lambda_WmK"],          # steel thermal conductivity [W/(m·K)]
        rho_wall                = row["steel_rho_kgm3"],            # steel density [kg/m³]
        cp_wall                 = row["steel_cp_JkgK"],             # steel specific heat [J/(kg·K)]
        # --- thermal: casing ---
        k_casing                = row["HDPE_lambda_WmK"],           # HDPE thermal conductivity [W/(m·K)]
        # --- burial ---
        depth                   = row["burial_depth_m"],            # burial depth [m]
        h_ext                   = row["h_ext_Wm2K"],                # external heat transfer coeff. [W/(m²·K)]
        # --- simulation ---
        stepsize                = STEPSIZE,                         # time step [s]
    )

    # Supply pipe
    net.add_lagrangian_pipe(
        name       = f"{name}_s",
        start_node = f"{frm}_s",
        end_node   = f"{to}_s",
        line       = "supply",
        **common,
    )

    # Return pipe
    net.add_lagrangian_pipe(
        name       = f"{name}_r",
        start_node = f"{to}_r",
        end_node   = f"{frm}_r",
        line       = "return",
        **common,
    )

print("--- Network setup complete. ---")

In [ ]:
# ---------------------------------------------------------------------------
# Schedules
# ---------------------------------------------------------------------------
# Consumers schedule
consumer_demand_schedule   = pd.DataFrame()
consumer_names = []

for i in node_data.index:
    row       = node_data.loc[i]
    name      = row["node_id"]
    node_type = row["node_type"]

    if node_type == "substation":
        consumer_names.append(name)

        q_wh = demand_profile_data["Q_W"] * (STEPSIZE / 3600)
        consumer_demand_schedule[name]   = -q_wh

consumer_demand_schedule   = consumer_demand_schedule[consumer_names]

# src1 schedule
src1_mdot_schedule = pd.DataFrame({src1_name: mdot_src1_ts})

schedules = {
    "heat_demand":  consumer_demand_schedule,     # [Wh]
    "setpoint_value_hyd": src1_mdot_schedule,     # [kg/s], src1
}

print(f"Schedules built: {len(consumer_names)} consumers × {NUM_SIM_STEPS} steps + "
      f"{(src1_name)} - auxiliary source(s) × {NUM_SIM_STEPS} steps")

In [ ]:
# ---------------------------------------------------------------------------
# Pre-heating (network initialization)
# ---------------------------------------------------------------------------
print("--- Initializing Network (Pre-heating for stability) ---")

INIT_STEPS = 144  # 1 day

init_demand_consumer = pd.DataFrame(
    np.tile(consumer_demand_schedule.iloc[0].values, (INIT_STEPS, 1)),
    columns=consumer_demand_schedule.columns,
)

init_mdot_src1 = pd.DataFrame(
    np.tile(src1_mdot_schedule.iloc[0].values, (INIT_STEPS, 1)),
    columns=src1_mdot_schedule.columns,
)

init_schedules = {
    "heat_demand":        init_demand_consumer,
    "setpoint_value_hyd": init_mdot_src1,
}

base_loop_init = SimpleStep(
    hydraulic_sim_kwargs={"error_threshold": 1e-6, "verbose": False},
    thermal_sim_kwargs={"error_threshold": 1e-6, "verbose": False},
    with_thermal=True,
)

scheduler_init = Scheduler(
    base_loop=base_loop_init,
    schedules=init_schedules,
    steps=INIT_STEPS,
)

import time
t0 = time.perf_counter()
_  = scheduler_init.execute(net=net, fluid=fluid, soil=soil)
print(f"Pre-heating done in {time.perf_counter()-t0:.2f} s. "
      f"({INIT_STEPS} steps = {INIT_STEPS*STEPSIZE/3600:.1f} h.)")

In [ ]:
# ---------------------------------------------------------------------------
# Full simulation
# ---------------------------------------------------------------------------
print("--- Running with Scheduler (Full Simulation) ---")

base_loop_scheduler = SimpleStep(
    hydraulic_sim_kwargs={"error_threshold": 1e-6, "verbose": False},
    thermal_sim_kwargs  ={"error_threshold": 1e-6, "verbose": False},
    with_thermal=True,
)

scheduler = Scheduler(
    base_loop=base_loop_scheduler,
    schedules=schedules,
    steps=NUM_SIM_STEPS,
)

t0 = time.perf_counter()
results = scheduler.execute(net=net, fluid=fluid, soil=soil)
sim_time = time.perf_counter() - t0

print(f"Simulation complete.")
print(f"Average simulation time: {sim_time:.2f} s.")

In [ ]:
# ---------------------------------------------------------------------------
# Extracted variables
# ---------------------------------------------------------------------------
res = results.to_dataframes()

substation_names = node_data[node_data["node_type"] == "substation"]["node_id"].tolist()
source_name      = node_data[node_data["node_type"] == "source"]["node_id"].tolist()
pipe_names       = pipe_data["pipe_id"].tolist()

# ── ETS ──────────────────────────────────────────────────────────────────────
T_sub_s  = res["nodes"]["temperature"][[f"{n}_s" for n in substation_names]]
T_sub_r  = res["nodes"]["temperature"][[f"{n}_r" for n in substation_names]]
mdot_sub = res["edges"]["mass_flow"][substation_names]
Q_sub_W  = res["edges"]["delta_q"][substation_names] / (STEPSIZE / 3600)  # [W]
dT_sub   = pd.DataFrame(
    T_sub_s.values - T_sub_r.values,
    columns=substation_names,
)  # [K]

# ── PRODUCER ─────────────────────────────────────────────────────────────────
T_src_s  = res["nodes"]["temperature"][[f"{n}_s" for n in source_name]]
T_src_r  = res["nodes"]["temperature"][[f"{n}_r" for n in source_name]]
mdot_src = res["edges"]["mass_flow"][source_name]
Q_src_W  = res["edges"]["delta_q"][source_name] / (STEPSIZE / 3600)  # [W]

# ── PIPE ─────────────────────────────────────────────────────────────────────
dp_pipe_s   = res["edges"]["delta_p"][[f"{n}_s"   for n in pipe_names]]   # [Pa]
dp_pipe_r   = res["edges"]["delta_p"][[f"{n}_r"   for n in pipe_names]]   # [Pa]
mdot_pipe_s = res["edges"]["mass_flow"][[f"{n}_s" for n in pipe_names]]   # [kg/s]
mdot_pipe_r = res["edges"]["mass_flow"][[f"{n}_r" for n in pipe_names]]   # [kg/s]
T_pipe_s    = res["edges"]["temperature"][[f"{n}_s" for n in pipe_names]] # [°C]
T_pipe_r    = res["edges"]["temperature"][[f"{n}_r" for n in pipe_names]] # [°C]
Q_loss_pipe_s = res["edges"]["delta_q"][[f"{n}_s" for n in pipe_names]] / (STEPSIZE / 3600)  # [W]
Q_loss_pipe_r = res["edges"]["delta_q"][[f"{n}_r" for n in pipe_names]] / (STEPSIZE / 3600)  # [W]

# ── JUNCTION NODES ───────────────────────────────────────────────────────────
exclude_nodes = (
    {f"{n}_s" for n in substation_names} |
    {f"{n}_r" for n in substation_names} |
    {f"{source_name}_s", f"{source_name}_r"}
)
junction_cols = [c for c in res["nodes"]["temperature"].columns
                 if c not in exclude_nodes]
T_junction = res["nodes"]["temperature"][junction_cols]

# ── ENERGY BALANCE ───────────────────────────────────────────────────────────
SKIP = 144   # steps to skip (1 day initialization)

Q_prod_MWh  =  Q_src_W.iloc[SKIP:].sum().sum() * (STEPSIZE / 3600) / 1e6
Q_cons_MWh  = -Q_sub_W.iloc[SKIP:].sum().sum() * (STEPSIZE / 3600) / 1e6
Q_loss_MWh  = -(Q_loss_pipe_s.iloc[SKIP:].sum().sum()
               + Q_loss_pipe_r.iloc[SKIP:].sum().sum()) * (STEPSIZE / 3600) / 1e6
eta_network =  Q_cons_MWh / Q_prod_MWh

print("=" * 50)
print("ENERGY BALANCE")
print("=" * 50)
print(f"  Produced:       {Q_prod_MWh:.3f} MWh")
print(f"  Consumed:       {Q_cons_MWh:.3f} MWh")
print(f"  Pipe losses:    {Q_loss_MWh:.3f} MWh")
print(f"  Network η:      {eta_network*100:.1f} %")
print("=" * 50)

# ---------------------------------------------------------------------------
# Save results
# ---------------------------------------------------------------------------
import os

results_dir = os.path.join("..", "..", "results")
os.makedirs(results_dir, exist_ok=True)

output_path = os.path.join(results_dir, "DHN_TG_4G_M_M_BASE_results.xlsx")

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    # ── ETS ──────────────────────────────────────────────────────────────────
    T_sub_s.iloc[SKIP:].to_excel(writer,  sheet_name="sub_T_supply_degC")
    T_sub_r.iloc[SKIP:].to_excel(writer,  sheet_name="sub_T_return_degC")
    dT_sub.iloc[SKIP:].to_excel(writer,   sheet_name="sub_dT_K")
    mdot_sub.iloc[SKIP:].to_excel(writer, sheet_name="sub_mdot_kgs")
    Q_sub_W.iloc[SKIP:].to_excel(writer,  sheet_name="sub_Q_W")

    # ── PRODUCER ─────────────────────────────────────────────────────────────
    T_src_s.iloc[SKIP:].to_excel(writer,  sheet_name="src_T_supply_degC")
    T_src_r.iloc[SKIP:].to_excel(writer,  sheet_name="src_T_return_degC")
    mdot_src.iloc[SKIP:].to_excel(writer, sheet_name="src_mdot_kgs")
    Q_src_W.iloc[SKIP:].to_excel(writer,  sheet_name="src_Q_W")

    # ── PIPE ─────────────────────────────────────────────────────────────────
    dp_pipe_s.iloc[SKIP:].to_excel(writer,       sheet_name="pipe_s_dp_Pa")
    dp_pipe_r.iloc[SKIP:].to_excel(writer,       sheet_name="pipe_r_dp_Pa")
    mdot_pipe_s.iloc[SKIP:].to_excel(writer,     sheet_name="pipe_s_mdot_kgs")
    T_pipe_s.iloc[SKIP:].to_excel(writer,        sheet_name="pipe_s_T_out_degC")
    T_pipe_r.iloc[SKIP:].to_excel(writer,        sheet_name="pipe_r_T_out_degC")
    Q_loss_pipe_s.iloc[SKIP:].to_excel(writer,   sheet_name="pipe_s_Qloss_W")
    Q_loss_pipe_r.iloc[SKIP:].to_excel(writer,   sheet_name="pipe_r_Qloss_W")

    # ── JUNCTION NODES ───────────────────────────────────────────────────────
    T_junction.iloc[SKIP:].to_excel(writer, sheet_name="junction_T_degC")

    # ── ENERGY BALANCE ───────────────────────────────────────────────────────
    pd.DataFrame({
        "Q_produced_MWh": [Q_prod_MWh],
        "Q_consumed_MWh": [Q_cons_MWh],
        "Q_losses_MWh":   [Q_loss_MWh],
        "eta_network":    [eta_network],
    }).to_excel(writer, sheet_name="energy_balance")

print(f"Results saved in: {os.path.abspath(output_path)}")